# 08 — BANKING77 Cleaning, Split & Instruction Export

**Dataset**: `PolyAI/banking77` (HuggingFace, pulled directly, no local raw file)
**Inputs**: HF `train` (10,003 rows) + `test` (3,080 rows)
**Outputs**:
- `data/banking77/processed/{train,val,test}.parquet` — canonical splits for all 3 models
- `data/banking77/instruction/{train,val,test}.jsonl` — Alpaca format (via
  `scripts/prepare_banking77_instruction_data.py`)
- `outputs/metrics/banking77_stats.json` — extended with a `cleaning` key
- `outputs/metrics/banking77_test_hash.txt` — sha256 of the test parquet

## Design notes from notebook 07

1. **Raw data is real** (not synthetic). We do NOT inject synthetic errors. The
   contrast with Bitext's injection-based cleaning demo is the narrative point.
2. **Real quality issues found** in raw text (counts from notebook 07): some leading/trailing whitespace, ~450 rows with internal multi-space, ~10 rows with embedded newlines, ~50 rows with non-ASCII characters (currency signs, accents), ~20 rows with text > 300 chars. No NaN and no mojibake. Notebook 07's dedup + overlap checks ran on *raw* text and came back clean — but those checks do NOT catch `(text, label)` collisions that only emerge after NFKC + whitespace normalisation. Section 13 of this notebook reports what the post-normalisation audit actually finds.
3. **Class imbalance ~5.3×** (min 35, max 187). Every class has ≥ 9 samples so
   a stratified 88/12 split on train places ≥ 1 row of every class in val.
4. **Flat 77-way classification** — no category hierarchy (unlike Bitext).

## Cleaning pipeline order (deterministic)

1. Strip leading/trailing whitespace on `text`
2. Collapse internal whitespace (`" ".join(s.split())`) on `text`
3. Unicode NFKC normalise `text`
4. Drop rows where `text` became empty after normalisation (shouldn't happen)
5. Deduplicate on `(text, label)` pair, keep first
6. Flag length outliers (informational; we do not drop — they are legitimate
   long queries from real users)

No category casing fix (BANKING77 has no `category` column). We do NOT touch
the label names — they come straight from HF's ClassLabel mapping and are
what the downstream models will have to predict verbatim.

## Split protocol

- Apply cleaning to the HF `train` split (10,003 rows).
- Stratified 88/12 split on `label`, `random_state=42` → ~8,800 train / ~1,200 val.
- Apply the same cleaning transformations to the HF `test` split (3,080 rows)
  so train / val / test have identical schema — but we do NOT resplit the test
  set. It is pinned exactly as HF releases it, cleaned identically, written to
  `test.parquet`, and sha256-hashed so every downstream model sees the same
  file bit-for-bit.


## 1. Imports, config, seed

In [1]:
from __future__ import annotations

import hashlib
import json
import random
import re
import subprocess
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# --- reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- paths ---
PROJECT_ROOT = Path("/home/mcaai/zh0038qi/customer-support-llm")
BANKING77_DIR = PROJECT_ROOT / "data" / "banking77"
PROCESSED_DIR = BANKING77_DIR / "processed"
INSTRUCTION_DIR = BANKING77_DIR / "instruction"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

for d in (PROCESSED_DIR, INSTRUCTION_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- display ---
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# --- split ratios ---
VAL_FRAC_OF_TRAIN = 0.12     # 88/12 split on the 10,003 HF train rows

# --- outlier flagging (informational, not applied as drop) ---
OUTLIER_CHAR_MAX = 300

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("SEED   :", SEED)


pandas : 2.3.3
numpy  : 1.26.4
SEED   : 42


## 2. Load BANKING77 from HuggingFace

In [2]:
ds = load_dataset("PolyAI/banking77")
label_names: list[str] = ds["train"].features["label"].names
N_CLASSES = len(label_names)

df_train_raw = ds["train"].to_pandas().copy()
df_test_raw  = ds["test"].to_pandas().copy()

# Map integer label -> intent name string (used by downstream classifiers).
# Keep BOTH columns in the output parquet: `label` (int) is handy for
# scikit-learn / DistilBERT, `label_name` (str) is used by the LLM pipeline
# and by the evaluator for confusion matrices.
df_train_raw["label_name"] = df_train_raw["label"].map(lambda i: label_names[i])
df_test_raw["label_name"]  = df_test_raw["label"].map(lambda i: label_names[i])

assert df_train_raw.shape == (10003, 3), f"Unexpected train shape: {df_train_raw.shape}"
assert df_test_raw.shape  == (3080, 3),  f"Unexpected test shape: {df_test_raw.shape}"
assert N_CLASSES == 77
assert set(df_train_raw["label_name"].unique()) == set(label_names)

print(f"Train raw shape : {df_train_raw.shape}")
print(f"Test  raw shape : {df_test_raw.shape}")
print(f"Classes         : {N_CLASSES}")
df_train_raw.head(3)


Train raw shape : (10003, 3)
Test  raw shape : (3080, 3)
Classes         : 77


,text,label,label_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived after 2 weeks?,11,card_arrival
2,I have been waiting over a week. Is the card still coming?,11,card_arrival


## 3. Audit: real (unsolicited) quality issues

Same audit function as notebook 07 — we run it here to capture a baseline
before-cleaning snapshot that the cleaning log can reference.


In [3]:
MOJIBAKE_PAT = re.compile(r"(?:Ã.|â€.|Â.|ï»¿)")


def audit(frame: pd.DataFrame) -> dict:
    t = frame["text"].fillna("").astype(str)
    return {
        "n_rows": int(len(frame)),
        "nan_text": int(frame["text"].isna().sum()),
        "nan_label": int(frame["label"].isna().sum()),
        "empty_or_ws_text": int((t.str.strip() == "").sum()),
        "exact_dup_rows": int(frame.duplicated().sum()),
        "dup_text_label": int(frame.duplicated(subset=["text", "label"]).sum()),
        "dup_text_only": int(frame.duplicated(subset=["text"]).sum()),
        "leading_trailing_ws": int((t != t.str.strip()).sum()),
        "internal_multi_space": int(t.str.contains(r"  +", regex=True).sum()),
        "has_tab": int(t.str.contains("\t", regex=False).sum()),
        "has_newline": int(t.str.contains("\n", regex=False).sum()),
        "non_ascii": int(t.str.contains(r"[^\x00-\x7f]", regex=True).sum()),
        "mojibake": int(t.str.contains(MOJIBAKE_PAT).sum()),
        "very_long_gt_cap": int((t.str.len() > OUTLIER_CHAR_MAX).sum()),
    }


audit_train_before = audit(df_train_raw)
audit_test_before  = audit(df_test_raw)

print("=== BANKING77 train — before cleaning ===")
for k, v in audit_train_before.items():
    print(f"  {k:22s}: {v}")
print()
print("=== BANKING77 test — before cleaning ===")
for k, v in audit_test_before.items():
    print(f"  {k:22s}: {v}")


=== BANKING77 train — before cleaning ===
  n_rows                : 10003
  nan_text              : 0
  nan_label             : 0
  empty_or_ws_text      : 0
  exact_dup_rows        : 0
  dup_text_label        : 0
  dup_text_only         : 0
  leading_trailing_ws   : 9
  internal_multi_space  : 454
  has_tab               : 0
  has_newline           : 10
  non_ascii             : 52
  mojibake              : 0
  very_long_gt_cap      : 20

=== BANKING77 test — before cleaning ===
  n_rows                : 3080
  nan_text              : 0
  nan_label             : 0
  empty_or_ws_text      : 0
  exact_dup_rows        : 0
  dup_text_label        : 0
  dup_text_only         : 0
  leading_trailing_ws   : 3
  internal_multi_space  : 100
  has_tab               : 0
  has_newline           : 3
  non_ascii             : 9
  mojibake              : 0
  very_long_gt_cap      : 3


## 4. Cleaning pipeline

Applied to train and test identically. Each step logs before/after row counts
so no row is silently dropped.

**Design choice: drop-vs-keep for short queries.** BANKING77 has no
`text < 10 chars` rows to begin with (checked in notebook 07), so no action.
Long outliers (> 300 chars) are flagged but kept — real user queries can be
legitimately long and dropping them would bias the dataset toward a "tidy"
distribution the production model wouldn't face.


In [4]:
log: list[dict] = []


def _record(split: str, step: str, before: int, after: int, note: str = "") -> None:
    dropped = before - after
    log.append({
        "split": split, "step": step,
        "rows_before": before, "rows_after": after,
        "dropped": dropped, "note": note,
    })
    print(f"  [{split:5s}][{step:22s}] {before:6d} -> {after:6d} (dropped {dropped}) {note}")


def _collapse_ws(s):
    if not isinstance(s, str):
        return s
    return " ".join(s.split())


def _nfkc(s):
    if not isinstance(s, str):
        return s
    return unicodedata.normalize("NFKC", s)


def clean_split(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    """Apply the deterministic cleaning pipeline to one split."""
    df = frame.copy().reset_index(drop=True)

    # 4.1 Strip leading/trailing + 4.2 collapse internal whitespace
    df["text"] = df["text"].map(_collapse_ws)
    print(f"  [{split:5s}][whitespace_collapse   ] applied ' '.join(s.split())")

    # 4.3 NFKC normalise
    df["text"] = df["text"].map(_nfkc)
    print(f"  [{split:5s}][unicode_nfkc          ] NFKC normalise")

    # 4.4 Drop rows that became empty
    b = len(df)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    _record(split, "drop_empty_after_norm", b, len(df), "normalisation left text empty")

    # 4.5 Dedup on (text, label)
    b = len(df)
    df = df.drop_duplicates(subset=["text", "label"], keep="first").reset_index(drop=True)
    _record(split, "dedup_text_label", b, len(df), "keep=first")

    return df


print(">> Cleaning train split")
df_train = clean_split(df_train_raw, "train")
print()
print(">> Cleaning test split")
df_test = clean_split(df_test_raw, "test")

print()
print(f"Final cleaned shapes: train={df_train.shape}  test={df_test.shape}")


>> Cleaning train split
  [train][whitespace_collapse   ] applied ' '.join(s.split())
  [train][unicode_nfkc          ] NFKC normalise
  [train][drop_empty_after_norm ]  10003 ->  10003 (dropped 0) normalisation left text empty
  [train][dedup_text_label      ]  10003 ->   9999 (dropped 4) keep=first

>> Cleaning test split
  [test ][whitespace_collapse   ] applied ' '.join(s.split())
  [test ][unicode_nfkc          ] NFKC normalise
  [test ][drop_empty_after_norm ]   3080 ->   3080 (dropped 0) normalisation left text empty
  [test ][dedup_text_label      ]   3080 ->   3079 (dropped 1) keep=first

Final cleaned shapes: train=(9999, 3)  test=(3079, 3)


## 5. Audit: after cleaning

Defect counts should be at or below the pre-cleaning level. We do NOT require
every count to drop to zero — `non_ascii` rows (e.g. currency symbols) are
legitimate and must be preserved; `very_long_gt_cap` rows are kept as a
flag, not a drop.


In [5]:
audit_train_after = audit(df_train)
audit_test_after  = audit(df_test)

rows = []
for k in audit_train_before:
    rows.append({
        "defect": k,
        "train_before": audit_train_before[k],
        "train_after":  audit_train_after[k],
        "test_before":  audit_test_before[k],
        "test_after":   audit_test_after[k],
    })
pd.DataFrame(rows)


,defect,train_before,train_after,test_before,test_after
0,n_rows,10003,9999,3080,3079
1,nan_text,0,0,0,0
2,nan_label,0,0,0,0
3,empty_or_ws_text,0,0,0,0
4,exact_dup_rows,0,0,0,0
5,dup_text_label,0,0,0,0
6,dup_text_only,0,0,0,0
7,leading_trailing_ws,9,0,3,0
8,internal_multi_space,454,0,100,0
9,has_tab,0,0,0,0


In [6]:
# Hard assertions: whitespace defects MUST be cleaned.
assert audit_train_after["leading_trailing_ws"] == 0
assert audit_train_after["internal_multi_space"] == 0
assert audit_train_after["has_tab"] == 0
assert audit_train_after["has_newline"] == 0
assert audit_test_after["leading_trailing_ws"] == 0
assert audit_test_after["internal_multi_space"] == 0
assert audit_test_after["has_tab"] == 0
assert audit_test_after["has_newline"] == 0

# Class coverage preserved
assert df_train["label"].nunique() == N_CLASSES, "Lost a class from train"
assert df_test["label"].nunique() == N_CLASSES, "Lost a class from test"
print("Whitespace cleaning verified. All 77 classes still present in both splits.")


Whitespace cleaning verified. All 77 classes still present in both splits.


## 6. Stratified 88/12 split of the cleaned train into train/val

Leaves the HF test split untouched — it becomes our canonical test set.


In [7]:
df_train_split, df_val_split = train_test_split(
    df_train,
    test_size=VAL_FRAC_OF_TRAIN,
    random_state=SEED,
    stratify=df_train["label"],
    shuffle=True,
)

df_train_split = df_train_split.reset_index(drop=True)
df_val_split   = df_val_split.reset_index(drop=True)
df_test_split  = df_test.reset_index(drop=True)  # HF test, cleaned only

total_check = len(df_train_split) + len(df_val_split)
assert total_check == len(df_train), f"Split lost rows: {total_check} vs {len(df_train)}"

print(f"Cleaned train: {len(df_train)}")
print(f"  -> train : {len(df_train_split):>5} ({len(df_train_split) / len(df_train):.2%})")
print(f"  -> val   : {len(df_val_split):>5} ({len(df_val_split) / len(df_train):.2%})")
print(f"Cleaned test (HF test, pinned) : {len(df_test_split):>5}")


Cleaned train: 9999
  -> train :  8799 (88.00%)
  -> val   :  1200 (12.00%)
Cleaned test (HF test, pinned) :  3079


## 7. Drop cross-split leakage

Notebook 07's leakage check ran on the *raw* train/test texts and found zero
identical-text overlap. After whitespace normalisation, a small number of
`(text, label)` pairs collide between train and test — they were different
only in leading newlines or internal double-spaces. These are **genuine
real-data leakage artefacts** present in the upstream BANKING77 release that
were masked by formatting noise.

**Policy**: the test set is the canonical pinned evaluation set shared across
all 3 downstream models, so we preserve it exactly and drop the colliding
rows from the TRAIN side instead. (Dropping from test would invalidate the
cross-session sha256 pin.) We also dedup train vs the stratified val side
for the same reason.

Count and identities of removed rows are logged.


In [8]:
def drop_leakage(train_df: pd.DataFrame, other_df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop rows from `train_df` whose (text, label) pair also appears in
    `other_df`. Returns (cleaned_train, n_dropped)."""
    other_pairs = set(zip(other_df["text"], other_df["label"]))
    mask_leak = [
        (t, l) in other_pairs for t, l in zip(train_df["text"], train_df["label"])
    ]
    n_dropped = sum(mask_leak)
    cleaned = train_df.loc[[not m for m in mask_leak]].reset_index(drop=True)
    return cleaned, n_dropped


# Inspect leakage BEFORE fix
pairs_train_pre = set(zip(df_train_split["text"], df_train_split["label"]))
pairs_val_pre   = set(zip(df_val_split["text"],   df_val_split["label"]))
pairs_test_pre  = set(zip(df_test_split["text"],  df_test_split["label"]))

leak_train_val_pre  = pairs_train_pre & pairs_val_pre
leak_train_test_pre = pairs_train_pre & pairs_test_pre
leak_val_test_pre   = pairs_val_pre & pairs_test_pre

print(f"(text, label) overlap train vs val  (before fix) : {len(leak_train_val_pre)}")
print(f"(text, label) overlap train vs test (before fix) : {len(leak_train_test_pre)}")
print(f"(text, label) overlap val   vs test (before fix) : {len(leak_val_test_pre)}")

# Show each leaked example for audit trail
def _show_leak(pairs_set: set, title: str) -> None:
    if not pairs_set:
        return
    print(f"\n  -- {title} --")
    for t, l in sorted(pairs_set):
        print(f"    label={l} ({label_names[l]})  text={t!r}")


_show_leak(leak_train_val_pre,  "train vs val")
_show_leak(leak_train_test_pre, "train vs test")
_show_leak(leak_val_test_pre,   "val vs test")


(text, label) overlap train vs val  (before fix) : 0
(text, label) overlap train vs test (before fix) : 6
(text, label) overlap val   vs test (before fix) : 1

  -- train vs test --
    label=49 (pin_blocked)  text='How do I unblock my PIN?'
    label=24 (country_support)  text="I don't live in the UK. Can I still get a card?"
    label=22 (compromised_card)  text="There are a few transaction that I don't recognize, I think someone managed to get my card details and use it."
    label=10 (card_acceptance)  text='What businesses accept this card?'
    label=10 (card_acceptance)  text='Where can I use my card?'
    label=21 (change_pin)  text='Which cash machines will allow me to change my PIN?'

  -- val vs test --
    label=3 (atm_support)  text='At which ATMs can I use this card?'


In [9]:
# Drop from train: rows that collide with val or test
b_train = len(df_train_split)
df_train_split, n_drop_tv = drop_leakage(df_train_split, df_val_split)
df_train_split, n_drop_tt = drop_leakage(df_train_split, df_test_split)
_record("train", "drop_leakage_vs_val",  b_train, len(df_train_split) + n_drop_tt,
        f"removed {n_drop_tv} (text,label) pairs found in val")
_record("train", "drop_leakage_vs_test", len(df_train_split) + n_drop_tt, len(df_train_split),
        f"removed {n_drop_tt} (text,label) pairs found in test")

# Drop from val: rows that collide with test
b_val = len(df_val_split)
df_val_split, n_drop_vt = drop_leakage(df_val_split, df_test_split)
_record("val", "drop_leakage_vs_test", b_val, len(df_val_split),
        f"removed {n_drop_vt} (text,label) pairs found in test")

# Verify
pairs_train = set(zip(df_train_split["text"], df_train_split["label"]))
pairs_val   = set(zip(df_val_split["text"],   df_val_split["label"]))
pairs_test  = set(zip(df_test_split["text"],  df_test_split["label"]))

leak_train_val  = pairs_train & pairs_val
leak_train_test = pairs_train & pairs_test
leak_val_test   = pairs_val & pairs_test

print(f"\n(text, label) overlap train vs val  (after fix) : {len(leak_train_val)}")
print(f"(text, label) overlap train vs test (after fix) : {len(leak_train_test)}")
print(f"(text, label) overlap val   vs test (after fix) : {len(leak_val_test)}")
assert not leak_train_val and not leak_train_test and not leak_val_test, "Leakage still present after fix"
print("No (text, label) leakage across splits.")


  [train][drop_leakage_vs_val   ]   8799 ->   8799 (dropped 0) removed 0 (text,label) pairs found in val
  [train][drop_leakage_vs_test  ]   8799 ->   8793 (dropped 6) removed 6 (text,label) pairs found in test
  [val  ][drop_leakage_vs_test  ]   1200 ->   1199 (dropped 1) removed 1 (text,label) pairs found in test

(text, label) overlap train vs val  (after fix) : 0
(text, label) overlap train vs test (after fix) : 0
(text, label) overlap val   vs test (after fix) : 0
No (text, label) leakage across splits.


## 8. Per-split sanity checks

- Every class present in every split.
- Per-class distribution deviation between train-split and overall train ≤ 2 pp.


In [10]:
labels_train = set(df_train_split["label"].unique())
labels_val   = set(df_val_split["label"].unique())
labels_test  = set(df_test_split["label"].unique())
all_labels   = set(range(N_CLASSES))

missing_train = all_labels - labels_train
missing_val   = all_labels - labels_val
missing_test  = all_labels - labels_test
print(f"classes missing from train : {len(missing_train)}")
print(f"classes missing from val   : {len(missing_val)}")
print(f"classes missing from test  : {len(missing_test)}")
assert not missing_train and not missing_val and not missing_test, \
    "A split is missing one or more classes"

# Per-class fraction deviation vs overall train fraction
overall_dist = df_train["label"].value_counts(normalize=True).sort_index()
dist_train = df_train_split["label"].value_counts(normalize=True).reindex(overall_dist.index, fill_value=0.0)
dist_val   = df_val_split["label"].value_counts(normalize=True).reindex(overall_dist.index, fill_value=0.0)
dev_train_max = float((dist_train - overall_dist).abs().max())
dev_val_max   = float((dist_val   - overall_dist).abs().max())
print(f"\nMax |train_split - overall_train| dist deviation : {dev_train_max:.4f}")
print(f"Max |val_split   - overall_train| dist deviation : {dev_val_max:.4f}")
assert dev_train_max < 0.02, "Train distribution deviates > 2pp from overall"
assert dev_val_max   < 0.02, "Val distribution deviates > 2pp from overall"


classes missing from train : 0
classes missing from val   : 0
classes missing from test  : 0

Max |train_split - overall_train| dist deviation : 0.0002
Max |val_split   - overall_train| dist deviation : 0.0011


## 9. Write processed parquet files

Each row carries three columns: `text`, `label` (int 0-76), `label_name` (str).
Keep the int label for sklearn/BERT compatibility and the string label for the
LLM pipeline.


In [11]:
train_path = PROCESSED_DIR / "train.parquet"
val_path   = PROCESSED_DIR / "val.parquet"
test_path  = PROCESSED_DIR / "test.parquet"

df_train_split.to_parquet(train_path, index=False)
df_val_split.to_parquet(val_path,   index=False)
df_test_split.to_parquet(test_path, index=False)

for p, d in [(train_path, df_train_split), (val_path, df_val_split), (test_path, df_test_split)]:
    size_mb = p.stat().st_size / 1e6
    print(f"Wrote {p.name:14s} shape={d.shape}  {size_mb:.2f} MB")

# Schema round-trip sanity
_readback = pd.read_parquet(test_path)
assert list(_readback.columns) == ["text", "label", "label_name"]
assert _readback.shape == df_test_split.shape
print()
print("Schema round-trip OK. First 2 rows of test.parquet:")
_readback.head(2)


Wrote train.parquet  shape=(8793, 3)  0.32 MB
Wrote val.parquet    shape=(1199, 3)  0.05 MB
Wrote test.parquet   shape=(3079, 3)  0.09 MB

Schema round-trip OK. First 2 rows of test.parquet:


,text,label,label_name
0,How do I locate my card?,11,card_arrival
1,"I still have not received my new card, I ordered over a week ago.",11,card_arrival


## 10. Hash the canonical test parquet

Write `outputs/metrics/banking77_test_hash.txt` so every downstream model can
verify it is evaluating on the identical file. This is the BANKING77 analogue
of the Bitext test pinning.


In [12]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


test_hash = sha256_file(test_path)
hash_path = METRICS_DIR / "banking77_test_hash.txt"
hash_path.write_text(f"sha256:{test_hash}\n", encoding="utf-8")
print(f"sha256:{test_hash}")
print(f"Wrote {hash_path}")


sha256:6b7f43ccbe394d73310fa8d23ac97cebf9ce1292e989bca5f6001c52d8e33ddc
Wrote /home/mcaai/zh0038qi/customer-support-llm/outputs/metrics/banking77_test_hash.txt


## 11. Extend `banking77_stats.json` with a `cleaning` / `post_cleaning` key

Notebook 07 wrote the EDA snapshot under the `eda` key. We load, extend with
`cleaning`, `post_cleaning`, and `split_sizes`, and write back — so a single
file holds the entire data pipeline's provenance for BANKING77.


In [13]:
def get_git_commit(root: Path) -> str:
    """Return the short commit hash at `root`, or 'unknown' if not a repo."""
    try:
        out = subprocess.check_output(
            ["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
            stderr=subprocess.DEVNULL,
        )
        return out.decode().strip()
    except Exception:
        return "unknown"


stats_path = METRICS_DIR / "banking77_stats.json"
payload = json.loads(stats_path.read_text()) if stats_path.exists() else {}

split_sizes = {
    "train": int(len(df_train_split)),
    "val":   int(len(df_val_split)),
    "test":  int(len(df_test_split)),
    "train_frac_of_cleaned_hf_train": round(len(df_train_split) / len(df_train), 4),
    "val_frac_of_cleaned_hf_train":   round(len(df_val_split)   / len(df_train), 4),
    "cleaned_hf_train_rows": int(len(df_train)),
    "cleaned_hf_test_rows":  int(len(df_test)),
}

split_label_balance = {
    name: frame["label_name"].value_counts().sort_index().to_dict()
    for name, frame in [("train", df_train_split), ("val", df_val_split), ("test", df_test_split)]
}

payload["cleaning"] = {
    "seed": SEED,
    "git_commit": get_git_commit(PROJECT_ROOT),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "val_frac_of_train": VAL_FRAC_OF_TRAIN,
    "outlier_char_max": OUTLIER_CHAR_MAX,
    "audit_before": {
        "train": audit_train_before,
        "test":  audit_test_before,
    },
    "audit_after": {
        "train": audit_train_after,
        "test":  audit_test_after,
    },
    "cleaning_log": log,
    "split_sizes": split_sizes,
    "split_label_balance": split_label_balance,
    "parquet_paths": {
        "train": str(train_path),
        "val":   str(val_path),
        "test":  str(test_path),
    },
    "test_set_sha256": f"sha256:{test_hash}",
}

with stats_path.open("w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2, ensure_ascii=False)

# Round-trip check
_check = json.loads(stats_path.read_text())
assert "eda" in _check and "cleaning" in _check
assert _check["cleaning"]["split_sizes"]["train"] == len(df_train_split)

print(f"Wrote {stats_path} ({stats_path.stat().st_size / 1024:.1f} KB)")
print(f"keys: {list(_check.keys())}")
print(f"git commit : {payload['cleaning']['git_commit']}")
print(f"timestamp  : {payload['cleaning']['timestamp_utc']}")


Wrote /home/mcaai/zh0038qi/customer-support-llm/outputs/metrics/banking77_stats.json (18.1 KB)
keys: ['eda', 'cleaning']
git commit : 02d32c3
timestamp  : 2026-04-19T12:34:19+00:00


## 12. Convert splits to Alpaca JSONL

Uses `scripts/prepare_banking77_instruction_data.py` (headless CLI) — the
notebook and any CI run invoke exactly the same code path.

Each JSONL row:

```json
{"instruction": "Classify the following customer banking query into its intent.",
 "input":       "<customer query>",
 "output":      "Intent: <intent_name>"}
```

The `"Intent:"` prefix is the response anchor that the Qwen QLoRA
`DataCollatorForCompletionOnlyLM` will key on, mirroring the Bitext pipeline.


In [14]:
script_path = SCRIPTS_DIR / "prepare_banking77_instruction_data.py"
assert script_path.exists(), f"Missing script: {script_path}"

# The script auto-detects the `label_name` column in our parquet files and
# uses it verbatim — no temp-staging dance needed.
cmd = [
    "python", str(script_path),
    "--input-dir", str(PROCESSED_DIR),
    "--output-dir", str(INSTRUCTION_DIR),
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
print("--- stdout ---")
print(proc.stdout)
print("--- stderr ---")
print(proc.stderr)
assert proc.returncode == 0, f"Script failed with code {proc.returncode}"

for split in ("train", "val", "test"):
    p = INSTRUCTION_DIR / f"{split}.jsonl"
    n = sum(1 for _ in p.open("r", encoding="utf-8"))
    size_mb = p.stat().st_size / 1e6
    print(f"{p.name:14s}  {n:>5} rows  {size_mb:.2f} MB")


Running: python /home/mcaai/zh0038qi/customer-support-llm/scripts/prepare_banking77_instruction_data.py --input-dir /home/mcaai/zh0038qi/customer-support-llm/data/banking77/processed --output-dir /home/mcaai/zh0038qi/customer-support-llm/data/banking77/instruction


--- stdout ---

--- stderr ---
2026-04-19 20:34:20,203 | INFO | prepare_banking77_instruction_data | input_dir=/home/mcaai/zh0038qi/customer-support-llm/data/banking77/processed output_dir=/home/mcaai/zh0038qi/customer-support-llm/data/banking77/instruction splits=['train', 'val', 'test'] limit=None
2026-04-19 20:34:20,283 | INFO | prepare_banking77_instruction_data | Wrote 8793 rows -> /home/mcaai/zh0038qi/customer-support-llm/data/banking77/instruction/train.jsonl (from /home/mcaai/zh0038qi/customer-support-llm/data/banking77/processed/train.parquet, limit=None)
2026-04-19 20:34:20,297 | INFO | prepare_banking77_instruction_data | Wrote 1199 rows -> /home/mcaai/zh0038qi/customer-support-llm/data/banking77/instruction/val.jsonl (from /home/mcaai/zh0038qi/customer-support-llm/data/banking77/processed/val.parquet, limit=None)
2026-04-19 20:34:20,319 | INFO | prepare_banking77_instruction_data | Wrote 3079 rows -> /home/mcaai/zh0038qi/customer-support-llm/data/banking77/instruction/test.

In [15]:
# Preview 3 rows from train.jsonl
train_jsonl = INSTRUCTION_DIR / "train.jsonl"
with train_jsonl.open("r", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if i >= 3:
            break
        obj = json.loads(line)
        print(f"--- row {i} ---")
        print(json.dumps(obj, indent=2, ensure_ascii=False))

# JSONL integrity
for split in ("train", "val", "test"):
    p = INSTRUCTION_DIR / f"{split}.jsonl"
    with p.open("r", encoding="utf-8") as fh:
        for ln, line in enumerate(fh, 1):
            try:
                json.loads(line)
            except json.JSONDecodeError as e:
                raise AssertionError(f"Invalid JSON at {p}:{ln} — {e}")
    print(f"{p.name:14s}: every line parses as valid JSON")


--- row 0 ---
{
  "instruction": "Classify the following customer banking query into its intent.",
  "input": "How do I get this damn virtual card to work?",
  "output": "Intent: virtual_card_not_working"
}
--- row 1 ---
{
  "instruction": "Classify the following customer banking query into its intent.",
  "input": "Can I set my own top-up limit?",
  "output": "Intent: top_up_limits"
}
--- row 2 ---
{
  "instruction": "Classify the following customer banking query into its intent.",
  "input": "Why was I charged this extra fee while doing a transfer?",
  "output": "Intent: transfer_fee_charged"
}
train.jsonl   : every line parses as valid JSON
val.jsonl     : every line parses as valid JSON
test.jsonl    : every line parses as valid JSON


## 13. Real-data quality issues discovered

This is the narrative anchor for the BANKING77 half of the project. Unlike
Bitext (notebook 02) where we **injected** synthetic errors and celebrated
detecting them, here every issue below is a genuine artefact of the upstream
release and is reported, not manufactured.

### A. Within-split `(text, label)` duplicates (visible only after NFKC)

| Split | Raw dupes | Post-NFKC dupes | Action            |
|-------|-----------|-----------------|-------------------|
| train |         0 |           **4** | dropped (keep=first) |
| test  |         0 |           **1** | dropped (keep=first) |

The raw-text dedup in notebook 07 came back clean. The 4 train and 1 test
duplicates only surface once leading/trailing whitespace is stripped and
stray internal multi-spaces collapse — i.e. the same user query submitted
twice with a trivial whitespace difference on one copy.

### B. Cross-split `(text, label)` leakage (also only visible post-NFKC)

| Leaked from | Leaked into | Rows | Action                                   |
|-------------|-------------|------|------------------------------------------|
| train       | test        |    6 | dropped from **train** (test preserved)  |
| val         | test        |    1 | dropped from **val**   (test preserved)  |
| train       | val         |    0 | —                                        |

Total leakage pairs: **7**. All surface after the same whitespace + NFKC
normalisation that uncovered the within-split dupes — on the raw strings
they looked distinct because one side carried a stray `\n` or double-space.

### C. Decision: why we dedupe test, and why we drop train/val side on leakage

**Test deduplication (3,080 → 3,079).** We apply the identical cleaning
pipeline to the test split — not to resplit or reshape it, but to keep its
schema and string form consistent with train/val so downstream models see
the same preprocessing everywhere. The within-test duplicate that emerges is
a genuine pair of identical `(text, label)` rows; counting the correct
prediction twice would silently inflate the per-sample metrics. Dropping it
(once) is the conservative, honest choice.

**Leakage side: drop train/val, preserve test.** The pinned `test.parquet`
is the cross-model comparison contract for this project (sha256-hashed, all
three models must evaluate on bit-identical bytes). If we dropped from the
test side instead, we would destabilise that contract. If we kept the
leaking train/val rows, a model would see those exact prompts at training
time and then score them at test time — a textbook train/test contamination.

**Quantifying the avoided inflation.** 7 leakage pairs × 1.0 accuracy
contribution / 3,080 test rows ≈ **0.23 pp** of upward bias avoided. That
is small in absolute terms but structurally important: it would sit
unevenly across intents (leakage isn't uniformly distributed over the 77
classes), so it would contaminate per-class F1 and confusion analyses in
ways a global-accuracy delta alone would hide.

### D. Text-length outliers — flagged, kept

~20 train rows exceed 300 chars. Real users write long queries; dropping
these would sanitise the evaluation distribution away from production
reality, so we flag-but-keep.

### E. Non-ASCII — preserved as-is

~50 train rows contain currency signs (£, €) and accented names. Legitimate
content; NFKC normalises where appropriate and the characters stay.

### F. Final split results

- Stratified 88/12 split on the cleaned HF `train` (post-dedup: 9,999 rows).
- HF `test` (post-dedup: 3,079 rows) preserved as-is beyond cleaning. No
  resplit, so `test.parquet` can be sha256-pinned across sessions.
- After cross-split leakage fix: train **8,793** / val **1,199** / test **3,079**.
- Zero `(text, label)` leakage between any pair of splits.
- Max per-class distribution deviation between train-split and overall
  train is < 2 pp (well within tolerance).

### G. Contrast with Bitext's pipeline (the PM-facing takeaway)

- Bitext notebook 02 **injected** ~11 % synthetic errors and showed clean
  detection / removal. A tidy demo — but the noise was manufactured.
- BANKING77 notebook 08 reports **genuine** upstream-release issues: within-
  split near-duplicates and cross-split leakage, both masked by whitespace
  until we normalise. Less showy, more honest — this is the noise profile a
  production system actually sees.
- Both datasets end up as canonical `{train, val, test}` parquet + Alpaca
  JSONL files, pinned by sha256, consumed by the same three training
  pipelines.

### H. Consumers (downstream, not yet adapted)

- `scripts/train_tfidf.py` (adapted) → `data/banking77/processed/*.parquet`
- `scripts/train_distilbert.py` (adapted) → `data/banking77/processed/*.parquet`
- `scripts/train_qwen_lora.py` (adapted) → `data/banking77/instruction/*.jsonl`
